# Chapter 3 — Prompt Engineering with MLflow

In [0]:
%pip install -U databricks-sdk mlflow
dbutils.library.restartPython()

## Optional: common configuration

Many snippets below call `mlflow.set_tracking_uri("databricks")` and `mlflow.set_registry_uri("databricks-uc")`.
Those are safe to call multiple times, so the notebook keeps the original snippets unchanged.


## Zero-shot prompting

A basic Unity Airways example might look like this:

In [0]:
template = """
You are a customer support assistant for Unity Airways.

Task: Answer the customer's question using only the information provided.
If key details are missing, ask exactly one clarifying question.
Do not invent fees, waivers, or exceptions.

Customer question: {{question}}

Write a concise answer (max 120 words).
"""

## Few-shot prompting

Keep the examples realistic. If your examples sound like they were written by a legal department that has never met a customer, the model learns that tone too. Here is an example related to Unity Airways support assistant:

In [0]:
template = """
You are a customer support assistant for Unity Airways.

Goal: Help the customer with flight changes/cancellations/refunds.
Rules:
- If key details are missing, ask exactly ONE clarifying question.
- Do not invent fees, waivers, or eligibility. If policy depends on fare rules and you do not have them, say what you need to confirm.
- Keep the final answer under 120 words.
- Use this structure:
  1) Answer / Guidance
  2) What I need from you (only if needed)

Customer question: {{question}}

Examples:
Example 1 (Direct answer)
Customer question: What are your customer support hours?
Expected response: Unity Airways customer support is available 24/7 for urgent travel issues. For general enquiries, response times may be slower during peak periods.

Example 2 (Ask for missing details)
Customer question: Can I change my flight to next week?
Expected response: You can often change a booking, but eligibility and any fees depend on your fare type and ticket rules. Can you share your booking reference?

Example 3 (Decline to speculate)
Customer question: Will I definitely get a full refund if I cancel today?
Expected response: I can’t confirm a full refund without checking your fare rules, because refund eligibility and any cancellation fees depend on your ticket type and booking conditions. Could you share your booking reference?

Now respond to the customer question using the same style and rules.
"""

## Reasoning prompts

You are not asking for a novel. You are asking for better judgment.Here is an example related to Unity Airways support assistant:

In [0]:
template = """
You are a customer support assistant for Unity Airways.

Goal: Provide a correct, policy-safe answer with minimal risk of overpromising.

Instructions:
1) First, think through what key information is missing that would change eligibility (fare type, booking channel, route/date, disruption type, booking reference).
2) If key details are missing, ask exactly ONE clarifying question. Do not answer beyond what is safe without those details.
3) If details are sufficient, answer directly.
4) Do not invent fees, waivers, or refund eligibility. If policy is unclear, say what you need to confirm.
5) Keep the final response under 120 words.

Internal reasoning (do not reveal):
- List missing key details (if any)
- Decide: ask one question OR answer
- Identify any statements that would be speculation and avoid them

Customer question: {{question}}

Output (customer-facing only):
- If clarifying is needed: ask exactly ONE question.
- Otherwise: provide the answer in 2–4 short sentences.
"""

## Self-correction and reflection

Here is a lightweight example:

In [0]:
template = """
You are a customer support assistant for Unity Airways.

Write the answer. Then verify:
- You did not invent fees, waivers, or exceptions.
- You asked one clarifying question if key details were missing.
- The answer is under 120 words.

Return only the final answer.

Question: {{question}}
"""

## Prompt engineering best practices

Here is a simple approach that works well for user messages:

In [0]:
template = """
You are a customer support assistant for Unity Airways.

Instructions:
- Answer using only the information in the customer message.
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or exceptions.

Customer message: {{question}}
Write a concise answer (max 120 words).
"""

## Prompt engineering best practices

If your application needs to parse the output, you should not rely on the model’s goodwill. You should define an output contract in the prompt and keep it simple. For example, if your UI needs both an answer and a follow-up question, a JSON structure can reduce ambiguity:

In [0]:
template = """
You are a customer support assistant for Unity Airways.

Return valid JSON with exactly these keys:
- "answer": string
- "clarifying_question": string or null

Rules:
- If key details are missing, put one question in "clarifying_question" and keep "answer" short.
- If nothing is missing, set "clarifying_question" to null.

Customer question: {{question}}
"""

## Naming prompts and choosing a location

Here is a simple setup pattern you can reuse:

In [0]:
import mlflow

# Configure MLflow tracking and choose an experiment for your project
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/unity-airways")

# Link this experiment to a default UC schema for prompts
mlflow.set_experiment_tags({
    "mlflow.promptRegistryLocation": "main.default"
})

## Creating prompts programmatically with the Python SDK

A Unity Airways-flavored example:

In [0]:
import mlflow

uc_prompt = "main.default.unity_airways_customer_support"

v1 = mlflow.genai.register_prompt(
    name=uc_prompt,
    template="""\
You are a customer support assistant for Unity Airways.

Rules:
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or exceptions.

Customer question: {{question}}

Write a concise answer (max 120 words).
""",
    commit_message="v1: baseline support answer with safety and brevity constraints",
    tags={
        "use_case": "customer_support",
        "language": "en",
        "owner": "unity-airways-support",
    },
)

print(f"Created prompt {v1.name} version {v1.version}")

## Versioning prompts

Here is what a “small, intentional change” looks like:

In [0]:
import mlflow

uc_prompt = "main.default.unity_airways_customer_support"

v2 = mlflow.genai.register_prompt(
    name=uc_prompt,
    template="""\
You are a customer support assistant for Unity Airways.

Rules:
- If the question is ambiguous, ask exactly one clarifying question.
- If the customer mentions refunds, do not promise eligibility without fare details.
- Do not invent fees, waivers, or exceptions.
- Keep the answer under 120 words.

Customer question:
{{question}}

Answer:
""",
    commit_message="v2: tighten ambiguity handling and refund safety posture",
    tags={
        "change_type": "behavior",
        "risk": "medium",
        "hypothesis": "reduces overconfident refund promises",
    },
)

print(f"Created version {v2.version} of {v2.name}")

## Promoting via aliases

Setting an alias is a simple operation:

In [0]:
import mlflow
mlflow.genai.set_prompt_alias(
    name="main.default.unity_airways_customer_support",
    alias="staging",
    version=2
)
mlflow.genai.set_prompt_alias(
    name="main.default.unity_airways_customer_support",
    alias="production",
    version=1
)

## Searching prompts in Unity Catalog

Here is the pattern Databricks demonstrates:

In [0]:
import mlflow

# Required format: list all prompts in a catalog.schema
all_prompts = mlflow.genai.search_prompts("catalog = 'main' AND schema = 'default'")

# Filter programmatically
ua_prompts = [p for p in all_prompts if "unity_airways" in p.name.lower()]
support_prompts = [p for p in all_prompts if p.tags.get("use_case") == "customer_support"]

## Deleting prompts

That means cleanup typically looks like this:

In [0]:
from mlflow import MlflowClient
client = MlflowClient()
prompt_name = "main.default.unity_airways_customer_support"
# Delete specific versions first (required for Unity Catalog)
client.delete_prompt_version(prompt_name, "1")
client.delete_prompt_version(prompt_name, "2")
# Then delete the prompt itself
client.delete_prompt(prompt_name)

## Loading a prompt by version

The URI form is easy to standardize:

In [0]:
import mlflow

prompt_name = "main.default.unity_airways_customer_support"
prompt_v2 = mlflow.genai.load_prompt(f"prompts:/{prompt_name}/2")

print(prompt_v2.name, prompt_v2.version)

## Loading a prompt by alias

The production path is defined by aliases. Databricks specifies the alias URI syntax as: prompts:/{catalog}.{schema}.{prompt_name}@{alias}.

In [0]:
import mlflow

prompt_name = "main.default.unity_airways_customer_support"
prompt_prod = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@production")

print(prompt_prod.name, prompt_prod.version)

## Loading a prompt by alias

Here is that pattern, adapted to Unity Airways:

In [0]:
import os
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

def load_runtime_prompt() -> object:
    prompt_alias = os.getenv("PROMPT_ALIAS", "production")
    prompt_uri = os.getenv("PROMPT_URI", "main.default.unity_airways_customer_support")
    uri = f"prompts:/{prompt_uri}@{prompt_alias}"
    return mlflow.genai.load_prompt(uri)

prompt = load_runtime_prompt()
print(f"Loaded {prompt.name} v{prompt.version}")

## Rendering template variables with format()

A small wrapper makes this safer:

In [0]:
from typing import Any, Dict

def render_prompt(prompt_obj, variables: Dict[str, Any]) -> str:
    try:
        return prompt_obj.format(**variables)
    except Exception as e:
        raise ValueError(
            f"Prompt formatting failed for '{getattr(prompt_obj, 'name', '<unknown>')}' "
            f"(version={getattr(prompt_obj, 'version', '<unknown>')}). "
            f"Provided keys: {sorted(list(variables.keys()))}"
        ) from e

text = render_prompt(prompt, {"question": "Do I get a refund if I miss my flight?"})

## Making the model call: a minimal Databricks pattern

Here is a minimal end-to-end call:

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

prompt = mlflow.genai.load_prompt("prompts:/main.default.unity_airways_customer_support@production")
content = prompt.format(question="Can I change my flight tomorrow?")
resp = client.chat.completions.create(
    model="databricks-gpt-oss-20b",
    messages=[{"role": "user", "content": content}],
    temperature=0.1,
    max_tokens=350,
)
answer = resp.choices[0].message.content
print(answer)

## Building a prompt evaluation dataset

Here is a dataset creation pattern that stores the dataset in Unity Catalog and populates it with a few initial cases:

In [0]:
import mlflow
import uuid
mlflow.set_tracking_uri("databricks")
CATALOG = "main"
SCHEMA = "default"
SUFFIX = uuid.uuid4().hex[:8]
EVAL_DATASET_NAME = f"{CATALOG}.{SCHEMA}.ua_support_eval_{SUFFIX}"
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name=EVAL_DATASET_NAME
)
evaluation_examples = [
    {
        "inputs": {
            "question": "My flight is tomorrow. Can I change it to next week?"
        },
        "expectations": {
            "expected_facts": [
                "Eligibility depends on fare rules or fare type",
                "May involve a change fee or fare difference",
                "Ask for booking reference or fare details if missing"
            ]
        }
    },
    {
        "inputs": {
            "question": "I missed my flight due to traffic. Do I get a refund?"
        },
        "expectations": {
            "expected_facts": [
                "Refund eligibility depends on fare rules",
                "Do not promise a refund without checking ticket conditions",
                "Provide next steps to verify eligibility"
            ]
        }
    },
    {
        "inputs": {
            "question": "My flight was canceled. Can I rebook for free?"
        },
        "expectations": {
            "expected_facts": [
                "Rebooking depends on disruption policy",
                "Ask for booking details if needed",
                "Avoid claiming blanket waivers without evidence"
            ]
        }
    },
]
eval_dataset = eval_dataset.merge_records(evaluation_examples)
print(f"Added {len(evaluation_examples)} examples to {EVAL_DATASET_NAME}")

## Creating prompt versions to compare

Here is a simple example of registering two versions under the same prompt name:

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

PROMPT_NAME = f"{CATALOG}.{SCHEMA}.unity_airways_support_{SUFFIX}"

v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template="Answer this Unity Airways customer question: {{question}}",
    commit_message="v1: minimal support prompt"
)

v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template="""You are a careful Unity Airways customer support assistant.

Rules:
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or refund eligibility.
- Keep the answer under 120 words.

Customer question:
{{question}}

Answer:
""",
    commit_message="v2: add safety constraints and brevity limit"
)

print(f"Created versions v{v1.version} and v{v2.version} for {PROMPT_NAME}")

## Running comparative evaluation

We will use Databricks Model Serving exclusively via the OpenAI-compatible client provided by the Databricks SDK.

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

def create_support_function(prompt_name: str, version: int):
    """
    Returns a predict_fn for a specific prompt version.
    The function returns a dict so evaluation can read outputs consistently.
    """
    def answer_question(question: str) -> dict:
        prompt = mlflow.genai.load_prompt(
            name_or_uri=f"prompts:/{prompt_name}/{version}"
        )
        formatted = prompt.format(question=question)

        resp = client.chat.completions.create(
            model="databricks-gpt-oss-20b",
            messages=[{"role": "user", "content": formatted}],
            temperature=0.1,
            max_tokens=350,
        )
        return {"response": resp.choices[0].message.content}

    return answer_question

## Running comparative evaluation

Next, run evaluation for each version. The “checks” you choose should be minimal at first. A good starting point is correctness against expected facts because it aligns directly with your dataset structure.

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

checks = [
    Correctness(),  # uses expected_facts in the dataset expectations
]

results = {}
for version in [v1.version, v2.version]:
    print(f"Evaluating version {version}")
    with mlflow.start_run(run_name=f"ua_support_v{version}_eval"):
        mlflow.log_param("prompt_name", PROMPT_NAME)
        mlflow.log_param("prompt_version", version)
        mlflow.log_param("eval_dataset", EVAL_DATASET_NAME)

        eval_results = mlflow.genai.evaluate(
            predict_fn=create_support_function(PROMPT_NAME, version),
            data=eval_dataset,
            scorers=checks,
        )

        results[f"v{version}"] = eval_results
        print(f"Correctness: {eval_results.metrics.get('correctness/mean', 0):.2f}")

## Compare programmatically

If you want a quick numeric summary in notebooks or CI-style checks, keep the programmatic comparison. Start simple with a single metric, then add more checks only when you have a clear reason.

In [0]:
print("\n=== Version Comparison ===")
for version_label, result in results.items():
    correctness = result.metrics.get("correctness/mean", 0)
    print(f"{version_label}: correctness={correctness:.2f}")

best = max(results.items(), key=lambda kv: kv[1].metrics.get("correctness/mean", 0))
print(f"\nBest version by correctness: {best[0]}")

## Step 1: Register a baseline prompt

Establish a simple, baseline prompt version to serve as the "current behavior" and stable reference point. This initial prompt needs minimum guardrails (e.g., do not speculate on fees, ask for missing details, keep it short). Once registered, the baseline is versioned and immutable, allowing meaningful comparison for future changes. Each improvement becomes a new, clearly committed version.

In [0]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

PROMPT_NAME = "main.default.unity_airways_support_answer"

baseline = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template="""\
You are a customer support assistant for Unity Airways.

Rules:
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or refund eligibility.
- Keep the answer under 120 words.

Customer question:
{{question}}

Answer:
""",
    commit_message="v1: baseline support answer prompt"
)

print(f"Baseline prompt: {baseline.name} v{baseline.version}")

## Step 2: Define a prediction function

This function loads a specific prompt version, formats it, and sends it to a model endpoint via the OpenAI-compatible client from the Databricks SDK.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

def predict_fn(question: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{baseline.version}")
    content = prompt.format(question=question)

    resp = client.chat.completions.create(
        model="databricks-gpt-oss-20b",
        messages=[{"role": "user", "content": content}],
        temperature=0.1,
        max_tokens=350,
    )
    return resp.choices[0].message.content

## Step 3: Provide training examples with expected facts

Training examples should look like the questions you actually get, not like the questions you wish customers would ask. Include edge cases where the model is tempted to guess.

In [0]:
train_data = [
    {
        "inputs": {"question": "My flight is tomorrow. Can I change it to next week?"},
        "expectations": {
            "expected_facts": [
                "Eligibility depends on fare rules or fare type",
                "May involve a change fee or fare difference",
                "Ask for booking reference or fare details if missing"
            ]
        }
    },
    {
        "inputs": {"question": "I missed my flight due to traffic. Do I get a refund?"},
        "expectations": {
            "expected_facts": [
                "Refund eligibility depends on fare rules",
                "Do not promise a refund without checking ticket conditions",
                "Provide next steps to verify eligibility"
            ]
        }
    },
    {
        "inputs": {"question": "My flight was canceled. Can I rebook for free?"},
        "expectations": {
            "expected_facts": [
                "Rebooking depends on disruption policy",
                "Ask for booking details if needed",
                "Avoid claiming blanket waivers without evidence"
            ]
        }
    },
]

## Step 4: Run optimization and inspect the candidate template

Run optimization and extract the candidate template:

In [0]:
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=train_data,
    prompt_uris=[baseline.uri],
    optimizer=GepaPromptOptimizer(reflection_model="databricks:/databricks-claude-sonnet-4-5"),
    scorers=[Correctness(model="databricks:/databricks-gpt-5")],
)

candidate = result.optimized_prompts[0]
print("=== Candidate optimized template ===")
print(candidate.template)

## Publishing versions and assigning aliases

The following snippet creates two versions of a Unity Airways support prompt and assigns aliases so staging points to the newer candidate while production remains on the known-good version.

In [0]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

PROMPT = "main.default.unity_airways_customer_support"

# Version 1: baseline
v1 = mlflow.genai.register_prompt(
    name=PROMPT,
    template="""\
You are a Unity Airways customer support assistant.
Customer question: {{question}}
""",
    commit_message="v1: minimal support prompt",
)

# Version 2: safer, more constrained behavior
v2 = mlflow.genai.register_prompt(
    name=PROMPT,
    template="""\
You are a careful Unity Airways customer support assistant.

Rules:
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or refund eligibility.
- Keep the answer under 120 words.

Customer question:
{{question}}

Answer:
""",
    commit_message="v2: add safety constraints and brevity limit",
)

# Aliases control rollout
mlflow.genai.set_prompt_alias(name=PROMPT, alias="staging", version=v2.version)
mlflow.genai.set_prompt_alias(name=PROMPT, alias="production", version=v1.version)

print(f"Staging -> v{v2.version}, Production -> v{v1.version}")

## End-to-end “thin slice”

The example below shows an end-to-end function that answers a customer question using whatever alias is configured in the environment.

In [0]:
import os
import mlflow
from databricks.sdk import WorkspaceClient

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

def answer_customer(question: str) -> str:
    prompt_name = os.getenv("PROMPT_URI", "main.default.unity_airways_customer_support")
    alias = os.getenv("PROMPT_ALIAS", "production")

    # Load the prompt by alias
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@{alias}")

    # Render the template variables
    content = prompt.format(question=question)

    # Call the model serving endpoint
    resp = client.chat.completions.create(
        model="databricks-gpt-oss-20b",
        messages=[{"role": "user", "content": content}],
        temperature=0.1,
        max_tokens=350,
    )
    return resp.choices[0].message.content

print(answer_customer("My flight is tomorrow. Can I change it to next week?"))

## Promoting to production and rolling back safely

First, promote by moving the production alias to the candidate version:

In [0]:
import mlflow

mlflow.genai.set_prompt_alias(
    name="main.default.unity_airways_customer_support",
    alias="production",
    version=v2.version
)

## Promoting to production and rolling back safely

If anything unexpected happens, rollback is equally simple. Move production back to the previous version:

In [0]:
mlflow.genai.set_prompt_alias(
    name="main.default.unity_airways_customer_support",
    alias="production",
    version=v1.version
)